# Lab 02: Understanding Embeddings — SOLUTION

**Goal:** Learn what embeddings are by generating them, comparing them, and seeing how similar meaning produces similar vectors.

**What you'll learn:**
- How to create an embedding model with HuggingFaceEmbeddings
- What an embedding vector looks like (list of numbers)
- How cosine similarity measures meaning closeness
- Why embeddings are the foundation of RAG

## Step 1: Create an embedding model

all-MiniLM-L6-v2 is a small, fast model (80 MB) that runs locally.
No API key needed — it runs on your machine.
It produces 384-dimensional vectors.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("Embedding model ready!")

In [ ]:
def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors (0 to 1)."""
    dot = sum(a * b for a, b in zip(vec1, vec2))
    norm1 = sum(a * a for a in vec1) ** 0.5
    norm2 = sum(b * b for b in vec2) ** 0.5
    return dot / (norm1 * norm2)

## Step 2: Embed a single text

`embed_query()` converts text into a vector (list of numbers).
Each number represents one dimension of meaning.

In [ ]:
text = "What is the refund policy?"
vector = embeddings.embed_query(text)

print(f"Text: '{text}'")
print(f"Vector dimensions: {len(vector)}")
print(f"First 10 values: {[round(v, 4) for v in vector[:10]]}")

## Step 3: Compare similar vs different texts

Similar meaning → similar vectors → high cosine similarity

Different meaning → different vectors → low cosine similarity

In [ ]:
texts = [
    "What is the refund policy?",
    "How do I return a product?",
    "What is the weather in Mumbai?",
]
vectors = [embeddings.embed_query(t) for t in texts]

print("--- Similarity Comparison ---")
print(f"'{texts[0]}'")
print(f"  vs '{texts[1]}'  → similarity: {cosine_similarity(vectors[0], vectors[1]):.4f}")
print(f"  vs '{texts[2]}'  → similarity: {cosine_similarity(vectors[0], vectors[2]):.4f}")

## Step 4: Embed multiple documents at once

`embed_documents()` is optimized for batches of text.

In [ ]:
documents = [
    "Refund within 30 days of purchase",
    "Free shipping on orders above Rs 500",
    "Contact support at help@unigps.in",
    "Return items in original packaging",
]
doc_vectors = embeddings.embed_documents(documents)
print(f"--- Batch Embedding ---")
print(f"Embedded {len(doc_vectors)} documents, {len(doc_vectors[0])} dims each")

## Step 5: Find the most relevant document

Given a query, find which document is most similar.
This is exactly what a vector store does internally!

In [ ]:
query = "How do I get my money back?"
query_vec = embeddings.embed_query(query)

print(f"--- Finding Most Relevant Document ---")
print(f"Query: '{query}'\n")

scores = []
for doc, doc_vec in zip(documents, doc_vectors):
    sim = cosine_similarity(query_vec, doc_vec)
    scores.append((sim, doc))
    print(f"  [{sim:.4f}] {doc}")

best = max(scores, key=lambda x: x[0])
print(f"\nBest match: '{best[1]}' (score: {best[0]:.4f})")

## TODO 1: Test with your own similar/different pairs

Try embedding pairs of sentences and check their similarity.

In [ ]:
pairs = [
    ("Python is a programming language", "Python is a snake"),
    ("Bangalore weather", "Weather in Bengaluru"),
    ("Machine learning", "Deep learning"),
]
print("--- Custom Similarity Pairs ---")
for a, b in pairs:
    va = embeddings.embed_query(a)
    vb = embeddings.embed_query(b)
    sim = cosine_similarity(va, vb)
    print(f"  [{sim:.4f}] '{a}' vs '{b}'")

## TODO 2: Build a mini search engine

Add more documents and find the best match for a query.
Try adding documents about different topics and see which
ones the search finds most relevant.

In [ ]:
my_docs = [
    "How to set up a Docker container for Python",
    "Kubernetes pod networking explained",
    "FastAPI tutorial for beginners",
    "PostgreSQL performance tuning tips",
    "React component lifecycle methods",
]
print("--- Mini Search Engine ---")
my_query = "I want to learn web development with Python"
my_doc_vecs = embeddings.embed_documents(my_docs)
my_query_vec = embeddings.embed_query(my_query)
print(f"Query: '{my_query}'\n")
for doc, vec in zip(my_docs, my_doc_vecs):
    sim = cosine_similarity(my_query_vec, vec)
    print(f"  [{sim:.4f}] {doc}")

## Key Takeaways

- Embeddings turn text into vectors that capture meaning
- Similar meaning → similar vectors (high cosine similarity)
- all-MiniLM-L6-v2 runs locally, no API key needed
- This is the foundation of RAG's similarity search!